In [ ]:
import numpy as np
from scipy.optimize import minimize

# ==========================================
# PASO 1 y 2: FUNCIÓN OBJETIVO Y VARIABLES
# ==========================================
# Nuestro vector de decisión será x, donde:
# x[0] = temperatura (T)
# x[1] = caudal volumétrico (Q)
# x[2] = concentración de A (CA)
# x[3] = concentración de B (CB)

# Datos del problema
CA_in = 5.1      # mol/L
V_R   = 10.0     # L

k01 = 1.287e12   # 1/h
k02 = 1.287e12   # 1/h
k03 = 9.043e9    # L/(mol·h)

E1_R = 9758.3    # K
E2_R = 9758.3    # K
E3_R = 8560.0    # K

# Restricciones de operación
X_min  = 0.50    # conversión mínima
CD_max = 0.80    # máximo subproducto D (mol/L)

def funcion_objetivo(x):
    T  = x[0]
    Q  = x[1]
    CB = x[3]

    # Producción de B
    FB = Q * CB

    # scipy minimiza, por eso usamos el negativo
    return -FB


# ==========================================
# PASO 3: RESTRICCIONES
# ==========================================

# Balance de A en forma residual:
# 0 = (CA_in - CA)/tau - k1*CA - 2*k3*CA^2
def restriccion_balance_A(x):
    T  = x[0]
    Q  = x[1]
    CA = x[2]

    tau = V_R / Q

    k1 = k01 * np.exp(-E1_R / T)
    k3 = k03 * np.exp(-E3_R / T)

    r1 = k1 * CA
    r3 = k3 * CA**2

    return (CA_in - CA)/tau - r1 - 2*r3


# Balance de B en forma residual:
# 0 = -CB/tau + k1*CA - k2*CB
def restriccion_balance_B(x):
    T  = x[0]
    Q  = x[1]
    CA = x[2]
    CB = x[3]

    tau = V_R / Q

    k1 = k01 * np.exp(-E1_R / T)
    k2 = k02 * np.exp(-E2_R / T)

    r1 = k1 * CA
    r2 = k2 * CB

    return -CB/tau + r1 - r2


# Conversión mínima:
# XA >= X_min
def restriccion_conversion(x):
    CA = x[2]
    XA = 1.0 - CA / CA_in
    return XA - X_min


# Subproducto máximo:
# CD <= CD_max
# CD = tau * k3 * CA^2
def restriccion_subproducto(x):
    T  = x[0]
    Q  = x[1]
    CA = x[2]

    tau = V_R / Q
    k3 = k03 * np.exp(-E3_R / T)

    CD = tau * k3 * CA**2
    return CD_max - CD


# Restricciones para el solver
restricciones = [
    {'type': 'eq',   'fun': restriccion_balance_A},
    {'type': 'eq',   'fun': restriccion_balance_B},
    {'type': 'ineq', 'fun': restriccion_conversion},
    {'type': 'ineq', 'fun': restriccion_subproducto}
]

# Límites físicos
limites = (
    (300.0, 500.0),   # T
    (90.0, 800.0),    # Q
    (0.0001, CA_in),  # CA
    (0.0001, CA_in)   # CB
)

# Punto inicial
x_inicial = [410.0, 600.0, 2.5, 1.0]


# ==========================================
# PASO 4: RESOLUCIÓN
# ==========================================
resultado = minimize(
    funcion_objetivo,
    x_inicial,
    method='SLSQP',
    bounds=limites,
    constraints=restricciones
)


# ==========================================
# PASO 5: MOSTRAR RESULTADOS
# ==========================================
print("\n--- RESULTADOS DE LA OPTIMIZACIÓN DEL REACTOR ---")

if resultado.success:
    T_opt  = resultado.x[0]
    Q_opt  = resultado.x[1]
    CA_opt = resultado.x[2]
    CB_opt = resultado.x[3]

    tau_opt = V_R / Q_opt

    k2_opt = k02 * np.exp(-E2_R / T_opt)
    k3_opt = k03 * np.exp(-E3_R / T_opt)

    CC_opt = tau_opt * k2_opt * CB_opt
    CD_opt = tau_opt * k3_opt * CA_opt**2
    XA_opt = 1.0 - CA_opt / CA_in
    FB_opt = Q_opt * CB_opt

    print(f"¡Solución encontrada en {resultado.nit} iteraciones!")
    print(f"Temperatura óptima: {T_opt:.2f} K")
    print(f"Caudal óptimo:      {Q_opt:.2f} L/h")
    print(f"Tiempo de residencia: {tau_opt*60:.2f} min")

    print("\nConcentraciones:")
    print(f"CA = {CA_opt:.4f} mol/L")
    print(f"CB = {CB_opt:.4f} mol/L")
    print(f"CC = {CC_opt:.4f} mol/L")
    print(f"CD = {CD_opt:.4f} mol/L")

    print("\nDesempeño:")
    print(f"Conversión de A: {XA_opt:.4f}")
    print(f"Producción de B: {FB_opt:.4f} mol/h ({FB_opt/60:.4f} mol/min)")

else:
    print("El solver no pudo encontrar una solución.")